# Homework 9 - ST - 554 Big data analysis

## Author: Yefrid Cordoba

In this homework, we will model the compressive strength of concrete based on the amount of each component per cubic meter of material(concrete). The dataset *Concrete Compressive Strength* from the UC Irvine Machine Learning Repository will be used. We will fit three models: a multiple linear regression (MLR), a lasso regression, and a random forest — and compare their performance using the root mean squared error (RMSE) to determine the best model.

We initialize the sparl session.

In [67]:
!pip install pyspark

In [68]:
import pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, Interaction, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [69]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()

### Splitting the data, metrics, and models

#### Split

The data is going to be imported as `pd.DataFrame`, and then turned into a sparkSQL dataframe

In [70]:
Concrete = pd.read_excel("Concrete_Data.xls", engine='xlrd')
Concrete = spark.createDataFrame(Concrete)
Concrete.columns

['Cement (component 1)(kg in a m^3 mixture)',
 'Blast Furnace Slag (component 2)(kg in a m^3 mixture)',
 'Fly Ash (component 3)(kg in a m^3 mixture)',
 'Water  (component 4)(kg in a m^3 mixture)',
 'Superplasticizer (component 5)(kg in a m^3 mixture)',
 'Coarse Aggregate  (component 6)(kg in a m^3 mixture)',
 'Fine Aggregate (component 7)(kg in a m^3 mixture)',
 'Age (day)',
 'Concrete compressive strength(MPa, megapascals) ']

Now we rename the columns

First we create a list with the new names for the variables, in this step we start preparing the column names for the use of MLlib which requires the response variable to be called `label`, while the vector with the explanatory variables or `features` is going to be created later.

In [71]:
names = ['cement', 'slag', 'ash', 'water', 'plasticizer', 'coarse', 'fine', 'age', 'strength']

In [72]:
Concrete = Concrete.toDF(*names)

In [73]:
Concrete.show(5)

+------+-----+---+-----+-----------+------+-----+---+------------------+
|cement| slag|ash|water|plasticizer|coarse| fine|age|          strength|
+------+-----+---+-----+-----------+------+-----+---+------------------+
| 540.0|  0.0|0.0|162.0|        2.5|1040.0|676.0| 28|       79.98611076|
| 540.0|  0.0|0.0|162.0|        2.5|1055.0|676.0| 28|61.887365759999994|
| 332.5|142.5|0.0|228.0|        0.0| 932.0|594.0|270|40.269535256000005|
| 332.5|142.5|0.0|228.0|        0.0| 932.0|594.0|365|      41.052779992|
| 198.6|132.4|0.0|192.0|        0.0| 978.4|825.5|360|      44.296075096|
+------+-----+---+-----+-----------+------+-----+---+------------------+
only showing top 5 rows


Here we are going to split the data between the training an test set for the models.

In [74]:
train, test = Concrete.randomSplit([0.8,0.2], seed = 12)

#### Metrics

The root mean squared error (`RMSE`) is the metric that will be used to judge all the models to predict the compressive strength of concrete.\
This metric is chosen because it penalizes larger deviations more heavily than smaller ones. `RMSE` is calculated by squaring the residuals, averaging them, and then taking the square root, a process that amplifies the influence of data points that deviate significantly from the predicted values, as shown in the formula below.

$
RMSE = \sqrt{\sum_{i=1}^n{\frac{(\hat{y}_i - y_i)^2}{n}}}
$

#### Models

##### MLR

The multiple linear regression is an extension of simple linear regression that account for the multivariate nature of data. Each of the coefficients $\beta_i$ represents how variable the response is with respect to their $x_i$, that is, with each of the betas.\
The loss function used to fit the model is the RMSE, which is also going to be the metric to compare and judge between the models.

$
y = \beta_0 + \beta_1x_1 + ... + \beta_kx_k + \epsilon
$

From this model, interaction terms will also be included in the evaluation to assess the fit of this model and compare it with regularized model (Lasso) and random forest and decision tree models.

##### Lasso

Lasso regression is a regularization technique that enhances linear regression by adding an $L_1$ (aka $\lambda$)penalty to the loss function. This penalty shrinks less important coefficients toward zero, effectively performing automated variable selection by zeroing out redundant or irrelevant predictors. This will reduce model complexity and prevent overfitting.

$
\sum_{i = 1}^n({y_i}-\sum_j{x_{ij}\beta_j})^2 + \lambda \sum_{j=1}^p{|\beta_j|}
$

##### Random forest

This method uses bootstrap aggregating and randomness to reduce the high variance from decision trees. Each tree is trained on a random sample of the data, ensuring the model sees a slightly different distribution. Also, the model only considers a random subset of predictors rather than the full set.\
The model aggregates the results, averaging the outputs for regression.

### Model fitting

#### MLR

In this model we are going to fit a model with two interaction terms `plasticizer*coarse` and `plasticizer*fine`, also we are going to create a transformation for the scaling of all of the variables before fitting the MLR, adn this is also going to be used for the lasso model.

First we are going to create the transformer for the columns that are going to be used in the interaction terms, since we can feed the interaction with vectors.

In [75]:
assemb1 = VectorAssembler(inputCols=['plasticizer'], outputCol='pl_vec')
assemb2 = VectorAssembler(inputCols=['coarse'], outputCol='co_vec')
assemb3 = VectorAssembler(inputCols=['fine'], outputCol='fi_vec')

Here we are going to create the interaction terms.

In [76]:
interact1 = Interaction(inputCols=['pl_vec', 'co_vec'], outputCol='inter_pl_co')
interact2 = Interaction(inputCols=['pl_vec', 'fi_vec'], outputCol='inter_pl_fi')

We test the transforms

In [77]:
interact2.transform(
    assemb3.transform(
        assemb1.transform(Concrete)
            )
                ).show(5)


+------+-----+---+-----+-----------+------+-----+---+------------------+------+-------+-----------+
|cement| slag|ash|water|plasticizer|coarse| fine|age|          strength|pl_vec| fi_vec|inter_pl_fi|
+------+-----+---+-----+-----------+------+-----+---+------------------+------+-------+-----------+
| 540.0|  0.0|0.0|162.0|        2.5|1040.0|676.0| 28|       79.98611076| [2.5]|[676.0]|   [1690.0]|
| 540.0|  0.0|0.0|162.0|        2.5|1055.0|676.0| 28|61.887365759999994| [2.5]|[676.0]|   [1690.0]|
| 332.5|142.5|0.0|228.0|        0.0| 932.0|594.0|270|40.269535256000005| [0.0]|[594.0]|      [0.0]|
| 332.5|142.5|0.0|228.0|        0.0| 932.0|594.0|365|      41.052779992| [0.0]|[594.0]|      [0.0]|
| 198.6|132.4|0.0|192.0|        0.0| 978.4|825.5|360|      44.296075096| [0.0]|[825.5]|      [0.0]|
+------+-----+---+-----+-----------+------+-----+---+------------------+------+-------+-----------+
only showing top 5 rows


Now we set up the `VectorAssembler()` for the for the features and non-scaled yet.

In [93]:
feature_cols = ['cement', 'slag', 'ash', 'water', 'plasticizer', 'coarse', 'fine', 'age',
                'inter_pl_co', 'inter_pl_fi']
f_assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="unscaled_features"
)

And set up the scaler for the variables

In [79]:
scaler = StandardScaler(
    inputCol="unscaled_features",
    outputCol="scaled_features",
    withStd=True,
    withMean=True
)

Then, it is set up a SQL transformer for the change of names to `feautures` and `label` to fit our models

In [80]:
sql_trans = SQLTransformer(
    statement="SELECT *, strength AS label, scaled_features AS features FROM __THIS__"
)

Last step, setting up the transformers, defines the linear regression model.

In [81]:
lr = LinearRegression()

Now that we have all set up, we can build the pipeline for this model.

In [82]:
mlr_pipeline = Pipeline(
    stages=[assemb1, assemb2, assemb3, interact1, interact2, f_assembler, scaler, sql_trans, lr]
)

Now that the pipeline is ready, we can use the train dataset to fit the MLR.

In [83]:
mlr_model = mlr_pipeline.fit(train)

Now we check the coefficients from the training dataset.

In [84]:
trained_lr = mlr_model.stages[-1]

coefficients = trained_lr.coefficients.toArray()
intercept = trained_lr.intercept

# Pair the estimated coefficients with the feature names using Pandas
coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": coefficients
})

print(f"Estimated Model Intercept: {intercept:.4f}\n")
print("Coefficients:")
print(coef_df.to_string(index=False))

Estimated Model Intercept: 35.4840

Coefficients:
    Feature  Coefficient
     cement    13.046521
       slag     9.414726
        ash     5.316855
      water    -2.326744
plasticizer   -14.875415
     coarse     0.255548
       fine     2.640448
        age     7.139668
inter_pl_co    18.570499
inter_pl_fi    -1.645084


#### Lasso

For this model we are going to use the same pipeline, but here we are going to use lasso regularization to do the filtering of estimators that are not important in the prediction for our model.

First we are going to define the lasso model, setting the `elasticnetParam = 1.0` to ensure no Ridge regression or elastic net is used during the fitting of this model.

In [85]:
lasso = LinearRegression(elasticNetParam=1.0)

Now we build the pipeline using the same transformation that was used before for the MLR model.

In [86]:
lasso_pipeline = Pipeline(
    stages=[assemb1, assemb2, assemb3, interact1, interact2, f_assembler, scaler, sql_trans, lasso]
)

Then we define the grod for the lasso penalty

In [87]:
paramGrid = ParamGridBuilder() \
    .addGrid(lasso.regParam, [0.001, 0.01, 0.1, 1.0, 10.0]) \
    .build()

And define our metric for the cross validation

In [88]:
evaluator = RegressionEvaluator(metricName="rmse")

The cross validator is set up in the next step

In [89]:
crossval = CrossValidator(
    estimator=lasso_pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=5,
    seed=12
)

And we fit the cross validator to the training set and obtain the parameters and the RMSE

In [90]:
cv_model = crossval.fit(train)

Now we get the best penalty and the coefficients for the lasso model

In [92]:
best_lasso_model = cv_model.bestModel.stages[-1]
best_reg_param = best_lasso_model.getOrDefault('regParam')
print(f"Best L_1: {best_reg_param}")

Best L_1: 0.001


In [94]:
coefficients = best_lasso_model.coefficients.toArray()
intercept = best_lasso_model.intercept
coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": coefficients
})
print(f"Estimated Model Intercept: {intercept:.4f}\n")
print("Coefficients:")
print(coef_df.to_string(index=False))

Estimated Model Intercept: 35.4840

Coefficients:
    Feature  Coefficient
     cement    13.151519
       slag     9.630426
        ash     5.560747
      water    -2.145016
plasticizer    -1.870869
     coarse     1.162080
       fine     3.152263
        age     7.262793
inter_pl_co    10.072584
inter_pl_fi    -6.104680


We can see that none of the parameters were droped from the model, however, the magnitude of the coefficients of the interaction terms is lower compared to the MLR model.

#### Random forest

For this model no scaling or inteaction terms are going to be included, since random forest will not be sensitive to this characteristics during the fitting of the model.\
Therefore, we are just going to create two transformations for this:
- one assemble the features in vector form



In [96]:
base_features = ['cement', 'slag', 'ash', 'water', 'plasticizer', 'coarse', 'fine', 'age']
base_assembler = VectorAssembler(inputCols=base_features, outputCol="features")

- and a second that change their names

In [97]:
base_sql_trans = SQLTransformer(
    statement="SELECT *, strength AS label FROM __THIS__"
)

We are going to define our Random Forest Regresson function.

In [95]:
rf = RandomForestRegressor(seed=12)

And build the pipeline using just the new transformations

In [98]:
rf_pipeline = Pipeline(stages=[base_assembler, base_sql_trans, rf])

Now for the oaramer grid we need to define the number of trees and the depth that is going to be evaluated during the cross validation.

In [100]:
paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [20, 50, 100]) \
    .addGrid(rf.maxDepth, [5, 10, 15]) \
    .build()
# and set up the evaluator RMSE
evaluator = RegressionEvaluator(metricName="rmse")

Before fitting the training set, we need to set up the cross validator for the random forest.

In [101]:
crossval_rf = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=5,
    seed=12
)

Now we can fit the crossval on the training dataset.

In [102]:
cv_rf_model = crossval_rf.fit(train)

We are going to show the best parameters for the model

In [103]:
best_rf = cv_rf_model.bestModel.stages[-1]

print(f"Best Number of Trees: {best_rf.getNumTrees}")
print(f"Best Max Depth: {best_rf.getOrDefault('maxDepth')}")


Best Number of Trees: 100
Best Max Depth: 15


And we check the importance for the on the features

In [105]:
importances = best_rf.featureImportances.toArray()

# Pair the importances with the original base feature names
importance_df = pd.DataFrame({
    "Feature": base_features,
    "Importance Score": importances
})

print("Feature Importances (Higher = More Predictive Power):")
print(importance_df.sort_values(by="Importance Score", ascending=False).to_string(index=False))

Feature Importances (Higher = More Predictive Power):
    Feature  Importance Score
        age          0.328004
     cement          0.215390
      water          0.148920
plasticizer          0.079339
       slag          0.068949
       fine          0.061824
     coarse          0.054226
        ash          0.043347


Based on the importance of each variable, age accounts for around 33% of the model's predictive power, followed by cement content at 22%. In comparison, ash and coarse aggregate content contribute only 4% and 5%, respectively.

### Model testing

In this section the three models are going to be evaluates using the selected metric `RMSE` and the best model will be selected.

We are going to generate the predictions for all the models using the test set, in this way we make sure that all the data is passing through the same pipeline as the training dataset.

In [106]:
mlr_preds = mlr_model.transform(test)
lasso_preds = cv_model.transform(test)
rf_preds = cv_rf_model.transform(test)

We calcualte the RMSE for each of the models

In [108]:
mlr_rmse = evaluator.evaluate(mlr_preds)
lasso_rmse = evaluator.evaluate(lasso_preds)
rf_rmse = evaluator.evaluate(rf_preds)

And compile everything in a dataframe for visualization and sort the valuse based on the RMSE.

In [112]:
comparison_df = pd.DataFrame({
    "Model": ["Multiple Linear Regression", "Lasso Regression", "Random Forest"],
    "Test RMSE": [mlr_rmse, lasso_rmse, rf_rmse]
})
comparison_df = comparison_df.sort_values(by="Test RMSE", ascending=True).reset_index(drop=True)
print(comparison_df)

                        Model  Test RMSE
0               Random Forest   4.838449
1  Multiple Linear Regression  10.381349
2            Lasso Regression  10.395452


Among the three models evaluated, the Random Forest demonstrated the best performance. This result is expected since Random Forest is a nonlinear ensemble model that tends to be more resilient when predicting new data. Its ability to average across multiple decision trees helps reduce variance and mitigate overfitting.